# 👕 Virtual Try-On Master Playground
**Supported Models:** IDM-VTON, CatVTON, OOTDiffusion, FitDiT, Any2Any, OutfitAnyone

--- 
Developed by **nano** | Integrated Virtual Try-On Solution

In [ ]:
#@title 🚀 1. Setup Core Environment
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!pip install xformers -r requirements.txt
!pip install diffusers transformers accelerate

# Install ComfyUI Manager
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git
print("✅ Core Setup Complete!")

In [ ]:
#@title 📦 2. Select & Install Models
MODEL_TYPE = "IDM-VTON" #@param ["IDM-VTON", "CatVTON", "OOTDiffusion", "FitDiT", "Any2Any", "OutfitAnyone"]

import os
!apt-get -y install -qq aria2

def install_idm_vton():
    %cd /content/ComfyUI/custom_nodes
    !git clone https://github.com/shadowcz007/ComfyUI-IDM-VTON.git
    !pip install -r ComfyUI-IDM-VTON/requirements.txt
    %cd /content/ComfyUI/models/checkpoints
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/yisol/IDM-VTON/resolve/main/unet/diffusion_pytorch_model.bin -o idm_vton_unet.bin

def install_cat_vton():
    %cd /content/ComfyUI/custom_nodes
    !git clone https://github.com/chflame163/ComfyUI_CatVTON_Wrapper.git
    !pip install -r ComfyUI_CatVTON_Wrapper/requirements.txt
    %cd /content/ComfyUI/models/checkpoints
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/zhy6769/CatVTON/resolve/main/catvton.safetensors

def install_oot_diffusion():
    %cd /content/ComfyUI/custom_nodes
    !git clone https://github.com/ZHO-ZHO-ZHO/ComfyUI-OOTDiffusion.git
    !pip install -r ComfyUI-OOTDiffusion/requirements.txt
    %cd /content/ComfyUI/custom_nodes/ComfyUI-OOTDiffusion/checkpoints
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/levihsu/OOTDiffusion/resolve/main/checkpoints/human_parsing.bin
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/levihsu/OOTDiffusion/resolve/main/checkpoints/openpose.bin

def install_fit_dit():
    %cd /content/ComfyUI/custom_nodes
    !git clone https://github.com/BoyuanJiang/FitDiT.git -b FitDiT-ComfyUI FitDiT
    !pip install -r FitDiT/requirements.txt
    !mkdir -p /content/ComfyUI/models/FitDiT_models
    %cd /content/ComfyUI/models/FitDiT_models
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/BoyuanJiang/FitDiT/resolve/main/pytorch_model.bin -o fitdit_model.bin

def install_any2any():
    %cd /content/ComfyUI/custom_nodes
    !git clone https://github.com/shadowcz007/ComfyUI-Any2Any.git
    %cd /content/ComfyUI/models/checkpoints
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Any2Any/Any2AnyTryon/resolve/main/any2any_tryon_model.bin

def install_outfit_anyone():
    print("Outfit Anyone often shares CatVTON wrapper logic...")
    install_cat_vton()
    %cd /content/ComfyUI/models/checkpoints
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/HumanAIGC/OutfitAnyone/resolve/main/outfit_anyone_weights.safetensors || echo "Manual weight download may be needed"

actions = {
    "IDM-VTON": install_idm_vton,
    "CatVTON": install_cat_vton,
    "OOTDiffusion": install_oot_diffusion,
    "FitDiT": install_fit_dit,
    "Any2Any": install_any2any,
    "OutfitAnyone": install_outfit_anyone
}

actions[MODEL_TYPE]()
print(f"\n✅ {MODEL_TYPE} Installed Successfully!")

In [ ]:
#@title 🌐 3. Launch ComfyUI (Cloudflare Tunnel)
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

import subprocess, threading, time, socket
def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        if sock.connect_ex(('127.0.0.1', port)) == 0: break
        sock.close()
    print("\nComfyUI Loaded! Click the link below:\n")
    p = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for line in p.stderr:
        l = line.decode()
        if "trycloudflare.com " in l: print("🔗 URL:", l[l.find("http"):])

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()
%cd /content/ComfyUI
!python main.py --dont-print-server